In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
# %% [code]
import sys
from pathlib import Path

# Add the project root (one level up from /notebooks) to the system path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Now import the engine from your src directory structure
from src.stress_engine.monte_carlo import run_comprehensive_stress_engine

# Execute the 4D parametric stress engine (5,000 paths across 81 nodes)
# This generates the parquet artifact directly in your data path
df_results = run_comprehensive_stress_engine(n_paths=5000, n_days=90)

# Display a preview of the generated stress matrix
display(df_results.head())

Initializing Unified Stress Engine: Evaluating 81 structural nodes (5000 paths each)...
Simulation complete in 2.78 seconds. Results saved to '../data/raw/synthetic_4d_monte_carlo_results.parquet'.


,Institution_Key,Volatility,Tail_Tier,Asymmetry,Shock_Severity,Mean_Max_DD,P95_Max_DD,Distress_Probability
0,Vol-Low_Tail-Fat_Asym-Low_Shock-Mild,Vol-Low,Tail-Fat,Asym-Low,Shock-Mild,-0.277284,-0.429510,0.0950
1,Vol-Low_Tail-Fat_Asym-Low_Shock-Mod,Vol-Low,Tail-Fat,Asym-Low,Shock-Mod,-0.278179,-0.434201,0.0944
2,Vol-Low_Tail-Fat_Asym-Low_Shock-Sev,Vol-Low,Tail-Fat,Asym-Low,Shock-Sev,-0.277678,-0.435453,0.0964
3,Vol-Low_Tail-Fat_Asym-Norm_Shock-Mild,Vol-Low,Tail-Fat,Asym-Norm,Shock-Mild,-0.279169,-0.437084,0.1000
4,Vol-Low_Tail-Fat_Asym-Norm_Shock-Mod,Vol-Low,Tail-Fat,Asym-Norm,Shock-Mod,-0.278661,-0.440652,0.1030


In [5]:
# %% [code]
import pandas as pd

# Load the consolidated 4D stress testing artifact
df_results = pd.read_parquet("../data/raw/synthetic_4d_monte_carlo_results.parquet")

# Inspect the top rows of the 81-node matrix
print(f"Total institutional nodes evaluated: {len(df_results)}")
print(df_results.head(10))

# Quick verification: Check distress probability variance across shock severity
summary_table = df_results.groupby("Shock_Severity")["Distress_Probability"].mean()
print("\nMean Distress Probability by Shock Severity Tier:")
print(summary_table)

Total institutional nodes evaluated: 81
                         Institution_Key Volatility  Tail_Tier  Asymmetry  \
0   Vol-Low_Tail-Fat_Asym-Low_Shock-Mild    Vol-Low   Tail-Fat   Asym-Low   
1    Vol-Low_Tail-Fat_Asym-Low_Shock-Mod    Vol-Low   Tail-Fat   Asym-Low   
2    Vol-Low_Tail-Fat_Asym-Low_Shock-Sev    Vol-Low   Tail-Fat   Asym-Low   
3  Vol-Low_Tail-Fat_Asym-Norm_Shock-Mild    Vol-Low   Tail-Fat  Asym-Norm   
4   Vol-Low_Tail-Fat_Asym-Norm_Shock-Mod    Vol-Low   Tail-Fat  Asym-Norm   
5   Vol-Low_Tail-Fat_Asym-Norm_Shock-Sev    Vol-Low   Tail-Fat  Asym-Norm   
6  Vol-Low_Tail-Fat_Asym-High_Shock-Mild    Vol-Low   Tail-Fat  Asym-High   
7   Vol-Low_Tail-Fat_Asym-High_Shock-Mod    Vol-Low   Tail-Fat  Asym-High   
8   Vol-Low_Tail-Fat_Asym-High_Shock-Sev    Vol-Low   Tail-Fat  Asym-High   
9  Vol-Low_Tail-Norm_Asym-Low_Shock-Mild    Vol-Low  Tail-Norm   Asym-Low   

  Shock_Severity  Mean_Max_DD  P95_Max_DD  Distress_Probability  
0     Shock-Mild    -0.277284   -0.429510     

In [ ]:
# %% [code]
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Set publication-grade styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "axes.labelsize": 10,
        "axes.titlesize": 11,
    }
)

# Load the consolidated 4D stress testing artifact
df_results = pd.read_parquet("../data/raw/synthetic_4d_monte_carlo_results.parquet")

# Inspect safely using display to avoid terminal/notebook text truncation
print(f"Total institutional nodes evaluated: {len(df_results)}")
display(df_results.head(10))

# Generate a faceted grid graph to visualize each combination instead of printing text tables
g = sns.catplot(
    data=df_results,
    x="Shock_Severity",
    y="Distress_Probability",
    hue="Asymmetry",
    col="Tail_Tier",
    row="Volatility",
    kind="bar",
    height=3.0,
    aspect=1.2,
    palette="deep",
)

g.set_axis_labels("Shock Severity Tier", "Probability of Distress (DD <= -40%)")
g.set_titles(col_template="Tail Tier: {col_name}", row_template="Vol: {row_name}")
g.add_legend(title="GJR Asymmetry")
g.fig.suptitle(
    "4D Stress Matrix: Distress Probability Across All Institutional Profiles",
    y=1.03,
    fontsize=13,
    fontweight="bold",
)

output_path = "../data/raw/stress_matrix_faceted_plot.png"
plt.savefig(output_path, bbox_inches="tight")
print(f"\nFaceted stress matrix graph successfully saved to '{output_path}'")
plt.show()